# Annotator comparison: DeepSeek V4.1-Flash vs V4-Pro

Two independent annotation runs over the same frozen manifests, using the same
frozen prompt (taxonomy `2026-09-21-v3`) and differing only in model. This
notebook reuses `scripts/compare_annotations.py` so the numbers here and on the
command line come from one implementation.

Two agreement measures, and nothing else:

- **strict** — the two `primary_label` values are identical.
- **lenient** — the `{primary, secondary}` sets share at least one label, so a
  domain one annotator ranked first and the other ranked second still counts.

Cohen's kappa is reported for the strict measure only: kappa needs exactly one
categorical choice per rater, which the lenient measure does not provide.

In [1]:
import importlib.util
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# Reuse the script rather than reimplementing the metrics.
spec = importlib.util.spec_from_file_location(
    'compare_annotations', ROOT / 'src' / 'protest_classifier' / 'cli' / 'compare_annotations.py'
)
ca = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ca)

RUNS = {
    'flash': ROOT / 'data/annotation_runs/deepseek-v4-1-flash/run-2026-09-21-flash-01',
    'pro': ROOT / 'data/annotation_runs/deepseek-v4-pro/run-2026-09-21-pro-01',
}
SPLITS = ['dev', 'test_locked', 'train_6000']

## Provenance

Both runs must share a prompt hash, or the comparison measures prompt drift rather than model difference.

In [2]:
import hashlib
import re

def provenance(name, run_dir):
    """Hash the prompt snapshot here rather than trusting what a run wrote about itself.

    The two runs used different provenance formats: run_metadata.json (flash)
    and provenance.md (pro), so read the model identity from either.
    """
    snapshot = run_dir / 'prompt_snapshot.md'
    digest = (hashlib.sha256(snapshot.read_bytes()).hexdigest()
              if snapshot.exists() else '')
    model = ''
    meta = run_dir / 'run_metadata.json'
    prov = run_dir / 'provenance.md'
    if meta.exists():
        data = json.loads(meta.read_text())
        model = data.get('model_identity_reported_by_runtime') or data.get('model_name', '')
    elif prov.exists():
        found = re.search(r'reported_model_identity:.*?`([^`]+)`', prov.read_text())
        model = found.group(1) if found else ''
    return {
        'annotator': name,
        'run': run_dir.name,
        'model': model,
        'prompt_sha256': digest[:16],
    }

SOURCE_SHA = hashlib.sha256((ROOT / 'docs/prompts/annotation_prompt.md').read_bytes()).hexdigest()
prov = pd.DataFrame([provenance(n, d) for n, d in RUNS.items()])

assert prov.prompt_sha256.nunique() == 1, 'Runs used different prompts; comparison would confound prompt with model'
assert prov.prompt_sha256.iloc[0] == SOURCE_SHA[:16], 'Run prompt differs from docs/prompts/annotation_prompt.md'
print(f'Both runs share the frozen prompt {SOURCE_SHA[:16]}... - the comparison isolates the model.')
prov

Both runs share the frozen prompt 19c2da75017c047b... - the comparison isolates the model.


,annotator,run,model,prompt_sha256
0,flash,run-2026-09-21-flash-01,deepseek/deepseek-flash,19c2da75017c047b
1,pro,run-2026-09-21-pro-01,deepseek-v4-pro,19c2da75017c047b


## Overview

One row per split.

In [3]:
results = {
    split: ca.compare(
        ca.load(ca.resolve(RUNS['flash'], split)),
        ca.load(ca.resolve(RUNS['pro'], split)),
    )
    for split in SPLITS
}

overview = pd.DataFrame([
    {
        'split': split,
        'n': r['n_compared'],
        'strict': r['strict_overlap'],
        'lenient': r['lenient_overlap'],
        'kappa': r['cohen_kappa_strict'],
        'lenient_gain': round(r['lenient_overlap'] - r['strict_overlap'], 4),
        'secondary_flash': r['n_secondary_a'],
        'secondary_pro': r['n_secondary_b'],
        'labels_used': r['n_labels_used'],
    }
    for split, r in results.items()
])
overview.style.format({
    'strict': '{:.2%}', 'lenient': '{:.2%}', 'kappa': '{:.4f}', 'lenient_gain': '{:+.2%}',
})

,split,n,strict,lenient,kappa,lenient_gain,secondary_flash,secondary_pro,labels_used
0,dev,497,90.74%,94.37%,0.8968,+3.63%,48,31,21
1,test_locked,840,91.31%,93.21%,0.9052,+1.90%,67,51,23
2,train_6000,6000,93.17%,95.25%,0.9256,+2.08%,699,475,23


Coverage check: every split should compare its full row count with nothing
left over on either side, otherwise the two runs did not annotate the same events.

In [4]:
pd.DataFrame([
    {'split': s, 'compared': r['n_compared'],
     'only_flash': r['only_in_a'], 'only_pro': r['only_in_b']}
    for s, r in results.items()
])

,split,compared,only_flash,only_pro
0,dev,497,0,0
1,test_locked,840,0,0
2,train_6000,6000,0,0


## Where the disagreement sits

Per-label Jaccard over primary labels: of the events either annotator called
this label, the share both did. Low values mark unsettled boundaries.

In [5]:
def weakest(split, n=8):
    frame = pd.DataFrame(results[split]['per_label']).head(n)
    return frame.rename(columns={'n_a': 'flash', 'n_b': 'pro'})

weakest('train_6000')

,label,flash,pro,agreed,jaccard
0,other,476,544,407,0.6639
1,"democratic institutions, corruption, elections...",178,153,140,0.7330
2,culture,62,67,55,0.7432
3,"crime, violence and victim justice",201,186,169,0.7752
4,unjust law enforcement,189,180,162,0.7826
5,racism,96,101,87,0.7909
6,civil liberties and censorship,93,89,83,0.8384
7,healthcare,189,197,177,0.8469


In [6]:
def disagreements(split, n=8):
    return (pd.DataFrame(results[split]['top_disagreements'])
            .rename(columns={'primary_label_a': 'flash', 'primary_label_b': 'pro'})
            .head(n)[['count', 'flash', 'pro']])

disagreements('train_6000')

,count,flash,pro
0,30,"democratic institutions, corruption, elections...",other
1,20,labor rights and wages,other
2,19,climate and environment,other
3,13,other,labor rights and wages
4,11,labor rights and wages,healthcare
5,11,other,climate and environment
6,11,"crime, violence and victim justice",other
7,11,education,pandemic


## Inspect one label

Set `FLASH_LABEL` to any label flash used and re-run the cell. It lists the
`train_6000` events where flash chose that label and pro chose something else,
with each side's secondary label, flash's evidence, and the note itself.

`ca.load` trims to the three comparison columns, so this reads the assembled
files directly to keep `notes` and `evidence`.

In [7]:
# --- pick a label, then re-run this cell -------------------------------------
FLASH_LABEL = 'climate and environment'
SPLIT = 'train_6000'
MAX_ROWS = 25
NOTE_CHARS = 200
# -----------------------------------------------------------------------------

def load_full(run_dir, split):
    frame = pd.read_csv(ca.resolve(run_dir, split), dtype=str).fillna('')
    keep = [c for c in ['event_id_cnty', 'notes', 'primary_label',
                        'alternative_labels', 'evidence'] if c in frame]
    return frame[keep]

pair = load_full(RUNS['flash'], SPLIT).merge(
    load_full(RUNS['pro'], SPLIT), on='event_id_cnty', suffixes=('_flash', '_pro')
)

available = sorted(pair.primary_label_flash.unique())
if FLASH_LABEL not in available:
    raise ValueError(f'{FLASH_LABEL!r} unused by flash in {SPLIT}. Available: {available}')

chosen = pair[pair.primary_label_flash == FLASH_LABEL]
split_off = chosen[chosen.primary_label_pro != FLASH_LABEL]
print(f'flash used {FLASH_LABEL!r} on {len(chosen)} events in {SPLIT}; '
      f'pro disagreed on {len(split_off)} ({len(split_off) / len(chosen):.1%}).')

print(f'\nwhere pro put them instead:')
display(split_off.primary_label_pro.value_counts().rename('count').to_frame())

detail = split_off.assign(note=split_off.notes_flash.str.slice(0, NOTE_CHARS)).rename(columns={
    'event_id_cnty': 'event',
    'primary_label_pro': 'pro',
    'alternative_labels_flash': 'flash_2nd',
    'alternative_labels_pro': 'pro_2nd',
    'evidence_flash': 'flash_evidence',
})[['event', 'pro', 'flash_2nd', 'pro_2nd', 'flash_evidence', 'note']]

print(f'\nfirst {min(MAX_ROWS, len(detail))} of {len(detail)}:')
detail.head(MAX_ROWS).style.hide(axis='index').set_properties(
    subset=['note', 'flash_evidence'], **{'white-space': 'normal', 'text-align': 'left'}
)

flash used 'climate and environment' on 675 events in train_6000; pro disagreed on 26 (3.9%).

where pro put them instead:


,count
primary_label_pro,
other,19
farmers,3
housing and rents,2
labor rights and wages,1
culture,1



first 25 of 26:


event,pro,flash_2nd,pro_2nd,flash_evidence,note
ITA20829,other,,,No Tav activists protest against the Turin-Lyon high-speed railway project,"On 1 January 2022, No Tav activists gathered outside the Turin-Lyon high-speed railway construction site in San Didero (Torino, Piemonte) to protest against the project."
NOR1793,other,,,oppose wind power development in the country,"On 17 September 2024, Motvind Norway activists demonstrated in Drammen, Viken, to oppose wind power development in the country. The demonstration was a part of the nationwide peaceful action staged in"
FRA20170,other,,,demonstrate against construction of the Montpellier LIEN road project,"On 13 November 2022, 100 people, including members of the collectives 'La Deroute des routes' and 'SOS Oulala', gathered in Grabels, to demonstrate against the construction of the Montpellier road pro"
FRA40797,farmers,,,protest a pesticide law over environmental impact and health risks,"On 28 June 2025, at the call of CP, farmers gathered and planted flowers in Cahors (Occitanie) to protest against the Duplomb Law, which aimed to legalize the use of certain previously banned pesticid"
FRA10304,other,,,Denounced the evacuation of the Engrenage gardens.,"On 21 July 2021, around 50 persons gathered at the Bareuzai square in Dijon. They denounced the evacuation of the Engrenage gardens the day before."
FRA20180,other,,,Protest against real estate and tourism projects densifying the Vercors plateau,"On 12 November 2022, around 300 people gathered in Villard-de-Lans, to denounce the multiplication of real estate and tourism projects that are densifying the Vercors plateau."
FRA40819,farmers,farmers,,"Farmers protested a law legalizing pesticides, citing environmental and health risks","On 28 June 2025, at the call of CP, farmers gathered and planted flowers in Figeac (Occitanie) to protest against the Duplomb Law, which aimed to legalize the use of certain previously banned pesticid"
ITA25199,other,,,protest against the tourist port expansion that would block public access to the sea; environmental groups involved,"On 23 November 2024, hundreds of people including fishers, gathered under the Ognina Bridge in Catania (Sicilia) to protest against the privatization of Borgo Marinaro, following the concession issued"
BIH1199,housing and rents,housing and rents,climate and environment,block a highway over flood damage linked to an illegal quarry; demand safety assessment and address housing issues,"On 30 October 2024, residents of Donja Jablanica blocked the M-17 highway in protest, expressing their dissatisfaction with the current state of infrastructure in their village. This protest followed"
BGR2822,other,,,protest against granting the local park embankment to a concession,"On 20 December 2022, residents of Velingrad protested against granting their local park embankment being to a concession, which would mean non-local investors would decide the fate of the Kleptuza lan"


## Reading it

Agreement is high and stable: kappa 0.90–0.93 across all three splits, which is
"almost perfect" on the Landis–Koch scale. Kappa sits just below raw overlap, so
the agreement is not an artefact of one dominant class.

Lenient scoring recovers a third to a half of strict disagreements. Those are
events where both annotators identified the same two domains and ranked them
differently, rather than events they read differently.

`other` is the main fault line. It is the weakest or near-weakest label on every
split and dominates the disagreement pairs in both directions — the
"no listed class fits" boundary is being drawn in different places. The two
classes v3 introduced, `crime, violence and victim justice` and
`civil liberties and censorship`, are the other soft spot; they have the least
settled boundary language in the prompt.

To compare any other pair of runs, point `RUNS` at their directories, or from
the command line:

```bash
uv run scripts/compare_annotations.py \
  --a <run-dir-or-csv> --b <run-dir-or-csv> \
  --name-a flash --name-b pro --splits dev test_locked train_6000
```